In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Function Vectors Replication Notebook

This notebook replicates the key experiments from the "Function Vectors in Large Language Models" paper.

## Overview

The paper investigates whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning.

## Key Components:
1. Load model and tokenizer
2. Load dataset and compute task-conditioned mean activations  
3. Compute function vector (FV) from top causal attention heads
4. Test FV intervention in different contexts (ICL, shuffled-label, zero-shot, natural text)

In [2]:
# Core imports
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Any
from sklearn.model_selection import train_test_split
from pathlib import Path

# Disable gradient computation for inference
torch.set_grad_enabled(False)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA H200 NVL
GPU Memory: 150.11 GB


In [3]:
# Utility function for setting seeds for reproducibility
def set_random_seed(seed: int) -> None:
    """Set seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

set_random_seed(42)
print("Seed set to 42")

Seed set to 42


## Dataset Utilities

Reimplementing the dataset loading and prompt creation utilities.

In [4]:
# Dataset class for ICL experiments
class ICLDataset:
    """Dataset class for in-context learning experiments with input-output pairs."""
    
    def __init__(self, data):
        if isinstance(data, str):
            self.data = pd.read_json(data)
        elif isinstance(data, dict):
            self.data = pd.DataFrame(data)
        else:
            self.data = data
        self.data = self.data[['input', 'output']]
    
    def __getitem__(self, idx):
        if isinstance(idx, int):
            return self.data.iloc[idx].to_dict()
        elif isinstance(idx, slice) or isinstance(idx, (list, np.ndarray)):
            return self.data.iloc[idx].to_dict(orient='list')
        elif isinstance(idx, str):
            return self.data[idx].tolist()
        raise ValueError(f"Invalid index type: {type(idx)}")
    
    def __len__(self):
        return len(self.data)
    
    def __repr__(self):
        return f"ICLDataset(features={self.data.columns.tolist()}, num_rows={len(self)})"


def split_dataset(dataset: ICLDataset, test_size: float = 0.3, seed: int = 42) -> Dict[str, ICLDataset]:
    """Split dataset into train, valid, and test sets."""
    train_data, valid_data = train_test_split(dataset.data, test_size=test_size, random_state=seed)
    test_data, valid_data = train_test_split(valid_data, test_size=test_size, random_state=seed)
    
    return {
        'train': ICLDataset(train_data.to_dict(orient='list')),
        'valid': ICLDataset(valid_data.to_dict(orient='list')),
        'test': ICLDataset(test_data.to_dict(orient='list'))
    }


def load_task_dataset(task_name: str, data_root: str = '/net/scratch2/smallyan/function_vectors_eval/dataset_files',
                      test_size: float = 0.3, seed: int = 32) -> Dict[str, ICLDataset]:
    """Load a task dataset from the dataset files."""
    for folder in ['abstractive', 'extractive']:
        file_path = os.path.join(data_root, folder, f'{task_name}.json')
        if os.path.exists(file_path):
            dataset = ICLDataset(file_path)
            return split_dataset(dataset, test_size=test_size, seed=seed)
    raise FileNotFoundError(f"Dataset {task_name} not found in {data_root}")


# Test dataset loading
dataset = load_task_dataset('antonym')
print(f"Train: {len(dataset['train'])}, Valid: {len(dataset['valid'])}, Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]}")

Train: 1678, Valid: 216, Test: 504
Sample: {'input': 'hardware', 'output': 'software'}


In [5]:
# Prompt construction utilities
def build_prompt_data(word_pairs: Dict[str, List[str]], 
                      query_pair: Optional[Dict[str, str]] = None,
                      add_bos: bool = True,
                      shuffle_outputs: bool = False,
                      prefixes: Dict[str, str] = None,
                      separators: Dict[str, str] = None) -> Dict:
    """
    Build prompt data structure for ICL experiments.
    """
    if prefixes is None:
        prefixes = {"input": "Q:", "output": "A:", "instructions": ""}
    if separators is None:
        separators = {"input": "\n", "output": "\n\n", "instructions": ""}
    
    # Add BOS token to instruction prefix if needed
    if add_bos:
        prefixes = {k: (v if k != 'instructions' else '<|endoftext|>' + v) 
                   for k, v in prefixes.items()}
    
    # Handle query pair
    if query_pair is not None:
        query_pair = {k: (v[0] if isinstance(v, list) else v) for k, v in query_pair.items()}
    
    # Build examples
    inputs = word_pairs.get('input', [])
    outputs = word_pairs.get('output', [])
    
    if shuffle_outputs and len(outputs) > 0:
        outputs = np.random.permutation(outputs).tolist()
    
    # Add space prefix to tokens for proper tokenization
    examples = [{'input': ' ' + str(inp), 'output': ' ' + str(out)} 
                for inp, out in zip(inputs, outputs)]
    
    query_with_space = None
    if query_pair:
        query_with_space = {k: ' ' + str(v) for k, v in query_pair.items()}
    
    return {
        'instructions': '',
        'prefixes': prefixes,
        'separators': separators,
        'examples': examples,
        'query_target': query_with_space
    }


def construct_prompt(prompt_data: Dict, query: str = None) -> str:
    """Construct the full ICL prompt string from prompt data."""
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    
    if isinstance(query, list):
        query = query[0]
    
    # Build primer (few-shot examples)
    prompt = prompt_data['prefixes']['instructions'] + prompt_data['instructions'] + prompt_data['separators']['instructions']
    
    for ex in prompt_data['examples']:
        prompt += prompt_data['prefixes']['input'] + ex['input'] + prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output'] + ex['output'] + prompt_data['separators']['output']
    
    # Add query
    prompt += prompt_data['prefixes']['input'] + query + prompt_data['separators']['input']
    prompt += prompt_data['prefixes']['output']
    
    return prompt


# Test prompt construction
test_pairs = dataset['train'][:5]
test_query = dataset['test'][21]
prompt_data = build_prompt_data(test_pairs, query_pair=test_query, add_bos=True)
prompt_str = construct_prompt(prompt_data)
print("ICL Prompt:")
print(repr(prompt_str))

ICL Prompt:
'<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:'


## Model Loading

Load GPT-J 6B model - the smallest model used in the original experiments.

In [6]:
# Model configuration and loading
def load_model_and_tokenizer(model_name: str, device: str = 'cuda'):
    """Load a language model and its tokenizer with proper configuration."""
    print(f"Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True).to(device)
    
    # Build model configuration
    if 'gpt-j' in model_name.lower():
        config = {
            'n_heads': model.config.n_head,
            'n_layers': model.config.n_layer,
            'resid_dim': model.config.n_embd,
            'name_or_path': model.config.name_or_path,
            'attn_hook_names': [f'transformer.h.{layer}.attn.out_proj' for layer in range(model.config.n_layer)],
            'layer_hook_names': [f'transformer.h.{layer}' for layer in range(model.config.n_layer)],
            'prepend_bos': False
        }
    else:
        raise NotImplementedError(f"Model {model_name} not yet supported")
    
    return model, tokenizer, config


# Load GPT-J
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_model_and_tokenizer(model_name)
print(f"\nModel config:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden dim: {model_config['resid_dim']}")

# Set the edit layer (typically around L/3 for GPT-J, which is layer 9)
EDIT_LAYER = 9
print(f"\nEdit layer for FV intervention: {EDIT_LAYER}")

Loading model: EleutherAI/gpt-j-6b


In [7]:
# Check if model loaded successfully
print(f"Model type: {type(model).__name__}")
print(f"Model device: {model.device}")
print(f"Model config verified - Layers: {model_config['n_layers']}, Heads: {model_config['n_heads']}")

In [8]:
# Model load check
print("Checking model...")
print(model.device)

In [9]:
# Simple test to see if kernel is responsive
1 + 1

In [10]:
print("hello")